In [28]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from scipy.stats import rankdata
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split

# Set random seeds for reproducible results
np.random.seed(42)
torch.manual_seed(42)

# ==========================================
# 1. LOAD DATA & SELECT COLUMNS
# ==========================================
raw = pd.read_csv("U_of_DE_Comp.csv")
df = raw.copy()
print(df.head())

# Core 18 derogatory & delinquency attributes
derog_cols = [
    'Del8', 'Del3', 'Del5', 'CO2', 'CO3', 'CO1', 'Col1',
    'BK2', 'BK3', 'BK1', 'PubRec1', 'Repo1', 'Del2',
    'Del4', 'Del9', 'Del1', 'Del6', 'Acct7'
]

# Behavioral stability & utilization attributes (breaks ties among clean accounts)
tie_breaker_cols = [c for c in ['Util1', 'Util2', 'Util3', 'Acct1', 'Acct5', 'Inq1'] if c in df.columns]
all_cols = derog_cols + tie_breaker_cols

X_preprocessed = df[all_cols].copy()


   Acct1  Acct2  Acct3  Acct4  Acct5  Acct6  Acct7  Acct8  Acct9  Acct10  ...  \
0     83      1      2      0    204      1     89      0      1       1  ...   
1     83      3     19      0    205     17     96      0      0       0  ...   
2     84      9     15      2    123     12     68      0      0       0  ...   
3     84      9     16      0    204     25     82      0      0       2  ...   
4     83      1      4      0    196      1    100      0      0       2  ...   

   line_12m line_24m  PubRec1  Repo1  Score1  Score2  Util1  Util2  Util3  \
0      1000     1000        0      0     715     689     40     60     40   
1     19500    19500        0      0     654     668     73     56     89   
2      1000     1000        0      0     627     626     75     76     81   
3     11000    11000        0      0     730     722     34     93     17   
4     10000    10000        0      0     794     808      5      8      6   

   Util4  
0     13  
1      9  
2     12  
3     

In [29]:
# ==========================================
# 2. CLEAN SENTINEL CODES & ALIGN RISK DIRECTION
# ==========================================
# Replace bureau missing/exclusion codes (95-99, 995-999, -1, -9) with NaN
# ==========================================
# 2. CLEAN SENTINEL CODES (RANGE-BASED) & ALIGN RISK DIRECTION
# ==========================================
for c in all_cols:
    if c == 'CO3':
        # CO3 is a dollar balance: remove negative values and 999990+ bureau codes
        X_preprocessed.loc[(X_preprocessed[c] < 0) | (X_preprocessed[c] >= 999990), c] = np.nan
    elif c in ['Acct1', 'Acct5', 'Del6']:
        # Month counters: anything < 0 or >= 900 (catches 996-999 AND 9990-9999) is a sentinel code
        X_preprocessed.loc[(X_preprocessed[c] < 0) | (X_preprocessed[c] >= 900), c] = np.nan
    elif c in ['Acct7', 'Util1', 'Util2', 'Util3']:
        # Percentage/ratio columns: anything < 0 or > 100 (catches 996, 9996, etc.) is a sentinel code
        X_preprocessed.loc[(X_preprocessed[c] < 0) | (X_preprocessed[c] > 100), c] = np.nan
    else:
        # Count columns (Del, CO, BK, Col, PubRec, Repo, Inq): anything < 0 or >= 90 is a sentinel code
        X_preprocessed.loc[(X_preprocessed[c] < 0) | (X_preprocessed[c] >= 90), c] = np.nan

# Handle Del6 (NaN now includes 9996 -> means never delinquent -> lowest risk)
X_preprocessed['Del6_never_delinquent'] = X_preprocessed['Del6'].isna().astype(float)
X_preprocessed['Del6'] = 120.0 - X_preprocessed['Del6'].fillna(120.0).clip(upper=120.0)

# Handle Acct7 (% never delinquent -> invert so higher % delinquent = higher risk)
X_preprocessed['Acct7'] = 100.0 - X_preprocessed['Acct7'].fillna(100.0)

# Invert protective stability metrics if present (shorter streak / younger file = higher risk)
for pos_col in ['Acct1', 'Acct5']:
    if pos_col in X_preprocessed.columns:
        max_val = X_preprocessed[pos_col].quantile(0.99)
        X_preprocessed[pos_col] = max_val - X_preprocessed[pos_col].fillna(max_val).clip(upper=max_val)

# Fill missing utilization ratios with column median if present
for util_col in ['Util1', 'Util2', 'Util3']:
    if util_col in X_preprocessed.columns:
        X_preprocessed[util_col] = X_preprocessed[util_col].fillna(X_preprocessed[util_col].median())

# Log-transform skewed count & dollar balance columns
no_log = {'Acct7', 'Del6', 'Del6_never_delinquent', 'Util1', 'Util2', 'Util3', 'Acct1', 'Acct5'}
for c in X_preprocessed.columns:
    if c not in no_log:
        X_preprocessed[c] = np.log1p(X_preprocessed[c].fillna(0).clip(lower=0))

# Standardize all features to zero mean and unit variance
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_preprocessed)

# ==========================================
# 3. DEFINE & TRAIN DEEP SVDD NEURAL NETWORK
# ==========================================
class DeepSVDD(nn.Module):
    def __init__(self, input_dim, rep_dim=8):
        super().__init__()
        # bias=False is required in Deep SVDD to prevent hypersphere collapse
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32, bias=False),
            nn.SiLU(),
            nn.Linear(32, 16, bias=False),
            nn.SiLU(),
            nn.Linear(16, rep_dim, bias=False)
        )

    def forward(self, x):
        return self.net(x)

X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
loader = DataLoader(TensorDataset(X_tensor), batch_size=256, shuffle=True)

model = DeepSVDD(input_dim=X_scaled.shape[1])
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

# Initialize the center point 'c' of the healthy hypersphere
model.eval()
with torch.no_grad():
    center = model(X_tensor).mean(dim=0)
    center[(abs(center) < 0.01) & (center < 0)] = -0.01
    center[(abs(center) < 0.01) & (center >= 0)] = 0.01

# Train network to pull normal profiles tight around 'c'
epochs = 50
model.train()
for epoch in range(epochs):
    epoch_loss = 0.0
    for (batch_x,) in loader:
        optimizer.zero_grad()
        embeddings = model(batch_x)
        dist = torch.sum((embeddings - center) ** 2, dim=1)
        loss = torch.mean(dist)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(batch_x)

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch + 1}/{epochs} - Hypersphere Distance Loss: {epoch_loss / len(X_tensor):.4f}")

# ==========================================
# 4. GENERATE 0-100 RISK SCORES & EXPORT
# ==========================================
model.eval()
with torch.no_grad():
    svdd_raw_dist = torch.sum((model(X_tensor) - center) ** 2, dim=1).numpy()

# Convert distances into a 0-100 percentile risk score (4 decimals to break ties at the top)
df['svdd_risk_score'] = np.round((rankdata(svdd_raw_dist, method='average') / len(svdd_raw_dist)) * 100, 4)

# Save full dataset with the new score column attached
df.to_csv("scored_credit_data_svdd.csv", index=False)
print("\nScoring complete! Saved to 'scored_credit_data_svdd.csv'.\n")

# ==========================================
# 5. PRINT TOP 10 HIGHEST-RISK ACCOUNTS
# ==========================================
display_cols = ['svdd_risk_score', 'Del8', 'Del3', 'Del5', 'CO2', 'CO3', 'Col1', 'BK1', 'BK2', 'BK3', 'PubRec1', 'Repo1', 'Del2', 'Del4', 'Del9', 'Del1', 'Del6', 'Acct7']
existing_display_cols = [c for c in display_cols if c in df.columns]

print("--- TOP 10 HIGHEST-RISK ACCOUNTS (DEEP SVDD) ---")
print(df.nlargest(10, 'svdd_risk_score')[existing_display_cols].to_string(index=False))


Epoch 10/50 - Hypersphere Distance Loss: 0.0000
Epoch 20/50 - Hypersphere Distance Loss: 0.0000
Epoch 30/50 - Hypersphere Distance Loss: 0.0000
Epoch 40/50 - Hypersphere Distance Loss: 0.0000
Epoch 50/50 - Hypersphere Distance Loss: 0.0000

Scoring complete! Saved to 'scored_credit_data_svdd.csv'.

--- TOP 10 HIGHEST-RISK ACCOUNTS (DEEP SVDD) ---
 svdd_risk_score  Del8  Del3  Del5  CO2   CO3  Col1  BK1  BK2  BK3  PubRec1  Repo1  Del2  Del4  Del9  Del1  Del6  Acct7
        100.0000     0     0     0    2   910     3    0    0    0        0      0     0     0     2     6  9996     25
         99.9997     0     0     0    2  1322     1    0    0    0        0      0     0     3     2     7  9996     36
         99.9993     2     0     0    1   156     0    1    1    0        0      0     0     3     0     5  9996     86
         99.9990     2     1     1    5  2387     5    3    1    0        0      0     1     6     2    15     1     38
         99.9987     2     0     0    0     0     0